## Data Engineering Module 3 - Delta Lake

### Loading CSV file into Unity Catalog Volume

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS spark_lab

In [0]:
import requests

# Define the current catalog
catalog_name = spark.sql("SELECT current_catalog()").collect()[0][0]

# Define the base path using the current catalog
volume_base = f"/Volumes/{catalog_name}/default/spark_lab"

# List of files to download
files = ["products.csv"]

# Download each file
for file in files:
    url = f"https://raw.githubusercontent.com/kuljotSB/DP-750/refs/heads/main/Databricks_Fundamentals/{file}"
    response = requests.get(url)
    response.raise_for_status()

    # Write to Unity Catalog volume
    with open(f"{volume_base}/{file}", "wb") as f:
        f.write(response.content)

### Loading Data into a Dataframe

In [0]:
df = spark.read.load(f'/Volumes/{catalog_name}/default/spark_lab/products.csv', format='csv', header=True)
display(df.limit(10))

### Load the Data into a Delta Table

### Storing in Unity Catalog Volume

In [0]:
delta_table_path = f"/Volumes/{catalog_name}/default/spark_lab/delta/products-delta" 
df.write.format("delta").mode("overwrite").save(delta_table_path)

In [0]:
df.dtypes

### Manipulating the Delta Table by creating a DeltaTable Object

In [0]:
from delta.tables import *
from pyspark.sql.functions import *

# Create a deltaTable object
deltaTable = DeltaTable.forPath(spark, delta_table_path)
# Update the table (reduce price of product 771 by 10%)
deltaTable.update(
   condition = "ProductID = '772'",
   #set = { "ListPrice": "ListPrice - 1.0 " })
   set={"ListPrice": (col("ListPrice").try_cast("int") + 100.01).try_cast("string")})
# View the updated data as a dataframe
#deltaTable.toDF().show(10)

deltaTable.toDF().filter("ProductID = '772'").show()
     

### Creating a Dataframe from the delta dataset

In [0]:
new_df = spark.read.format("delta").load(delta_table_path)
new_df.show()

### Explore Logging for the Delta Table

In [0]:
deltaTable.history(10).show(10, False, True)

### Creating a Data Catalog Table

In [0]:
df.write.format("delta").mode("ignore").saveAsTable("default.ProductsManaged")

### Accessing the Data Catalog Table

In [0]:
%sql
USE default;
SELECT * FROM ProductsManaged;